# Chapter 23: Circles and Cycles

**Source span.** Perspectives on Projective Geometry, Chapter 23, Sections 23.1-23.5; printed pages 443-464; PDF pages 465-486. The source was used for orientation and coverage only. The prose, code, diagrams, checks, and artifact design below are original.

**Chapter goal.** Build a working model of what a circle means when distance is not primitive but is induced by a Cayley-Klein absolute. By the end, the reader should be able to compute the conic representing a distance circle, see how it touches the fundamental conic, recognize limit cycles, and verify why Galilean cycles are parabolas while Galilean distance circles are only pairs of vertical lines.

**Chapter question.** When distance comes from an absolute conic, what should count as a circle?

The chapter starts from the most literal definition: fix a center `m`, fix a distance value, and collect all points `p` with that distance from `m`. In a Cayley-Klein plane, that condition is encoded by the primal and dual matrices `A` and `B` of the fundamental conic. If `M` is the skew matrix for the cross product `m x p`, then a distance circle sits in the conic pencil

`p.T @ (lambda * M.T @ B @ M + mu * A) @ p = 0`.

This notebook follows that formula as data. It does not draw a Euclidean circle and then reinterpret it. It builds the conic matrix from the absolute, checks the limiting members of the pencil, records the contact residuals with the fundamental conic, and then moves to the degenerate Galilean case where the familiar definitions separate.

In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-23-circles-and-cycles"
ARTIFACT_ROOT

## Computational Translation Guide

The source chapter uses several ideas that become compact matrix operations in the notebook.

| Book idea | Computational object | What we inspect |
| --- | --- | --- |
| Fundamental conic | A symmetric matrix `A`, with dual matrix `B` | The absolute is the measuring instrument, not a drawn background decoration. |
| Centered circle | The conic pencil `lambda M.T B M + mu A` | Moving through the pencil interpolates between zero radius and infinite radius. |
| Exceptional points | Points where the polar of `m` meets the absolute | Every finite circle with center `m` has double contact there. |
| Center at infinity | A limiting homogeneous center `m=(a,b,c)` with `c -> 0` | Euclidean circles flatten to a finite line plus the line at infinity. |
| Cycle | A circle-like object stable under limiting, dual, or curvature definitions | The right definition depends on which geometry degenerates. |
| Galilean circle | Points with the same Galilean distance from a finite point | A pair of vertical lines, not a parabola. |
| Galilean cycle | A limit of conics through the coalescing ideal points | A vertical parabola, with the line-pair circles as special degenerate cycles. |

**Library routing.** The conics are two-dimensional and algebraic, so Matplotlib is used for durable labeled diagrams, Plotly is used for the one parameter-deformation lab that benefits from a slider, SymPy is used for exact Galilean theorem checks, NetworkX is used for the organizing-principles dependency diagram, and the course-local projective/conic utilities handle homogeneous coordinates and artifact display.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import display

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_json,
    save_table,
)
from utils.projective import cross_ratio, hpoint

ensure_artifact_root(ARTIFACT_ROOT)
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
})

A_HYP = np.diag([1.0, 1.0, -1.0])
B_HYP = np.linalg.inv(A_HYP)
A_EUC = np.diag([0.0, 0.0, 1.0])
B_EUC = np.diag([1.0, 1.0, 0.0])
ARTIFACTS = []
CHECKS = {}


def skew_cross_matrix(m):
    x, y, z = np.asarray(m, dtype=float)
    return np.array([[0.0, z, -y], [-z, 0.0, x], [y, -x, 0.0]], dtype=float)


def circle_pencil_member(A, B, m, lam=1.0, mu=0.0):
    M = skew_cross_matrix(m)
    return lam * (M.T @ B @ M) + mu * A


def circle_through_point(A, B, m, p):
    M = skew_cross_matrix(m)
    zero_radius = M.T @ B @ M
    return float(p @ A @ p) * zero_radius - float(p @ zero_radius @ p) * A


def normalize_matrix(C):
    C = np.asarray(C, dtype=float)
    nrm = np.linalg.norm(C)
    if nrm == 0:
        return C.copy()
    C = C / nrm
    pivot = C[np.unravel_index(np.argmax(np.abs(C)), C.shape)]
    return C if pivot >= 0 else -C


def conic_grid(C, xlim=(-1.4, 1.4), ylim=(-1.4, 1.4), samples=420):
    xs = np.linspace(xlim[0], xlim[1], samples)
    ys = np.linspace(ylim[0], ylim[1], samples)
    xx, yy = np.meshgrid(xs, ys)
    zz = C[0, 0] * xx**2 + C[1, 1] * yy**2 + C[2, 2]
    zz += 2 * C[0, 1] * xx * yy + 2 * C[0, 2] * xx + 2 * C[1, 2] * yy
    return xs, ys, xx, yy, zz


def draw_conic(ax, C, xlim, ylim, color, label=None, lw=2.0, alpha=1.0, linestyle="-"):
    _, _, xx, yy, zz = conic_grid(C, xlim, ylim)
    cs = ax.contour(xx, yy, zz, levels=[0], colors=[color], linewidths=lw, linestyles=linestyle, alpha=alpha)
    proxy = None
    if label:
        proxy = plt.Line2D([0], [0], color=color, lw=lw, linestyle=linestyle, label=label)
    return cs, proxy


def draw_unit_absolute(ax, color="#222222", lw=1.8):
    t = np.linspace(0, 2 * np.pi, 600)
    ax.plot(np.cos(t), np.sin(t), color=color, lw=lw, label="fundamental conic")
    ax.set_aspect("equal", adjustable="box")


def draw_line(ax, line, xlim, color="#666666", lw=1.2, linestyle="--", label=None):
    a, b, c = np.asarray(line, dtype=float)
    xs = np.linspace(xlim[0], xlim[1], 200)
    if abs(b) > 1e-12:
        ys = -(a * xs + c) / b
        ax.plot(xs, ys, color=color, lw=lw, linestyle=linestyle, label=label)
    elif abs(a) > 1e-12:
        ax.axvline(-c / a, color=color, lw=lw, linestyle=linestyle, label=label)


def unit_circle_line_intersections(line):
    a, b, c = np.asarray(line, dtype=float)
    den = a * a + b * b
    foot = np.array([-a * c / den, -b * c / den], dtype=float)
    gap = max(0.0, 1.0 - float(foot @ foot))
    direction = np.array([-b, a], dtype=float) / math.sqrt(den)
    return [np.array([*(foot + s * math.sqrt(gap) * direction), 1.0]) for s in (-1.0, 1.0)]


def tangent_parallel_residual(C, A, point):
    left = C @ point
    right = A @ point
    denom = max(np.linalg.norm(left) * np.linalg.norm(right), 1e-12)
    return float(np.linalg.norm(np.cross(left, right)) / denom)


def save_current_figure(path):
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    ARTIFACTS.append(path)
    return path

## Source-Specific Storyboard

The storyboard below is the implementation plan for this chapter. Each visual has a mathematical job: expose a conic pencil, show a contact invariant, make a limiting process visible, classify the competing meanings of cycle, or verify a Galilean theorem by computation.

In [ ]:
STORYBOARD = [
    {"concept": "circles via Cayley-Klein distances", "representation": "conic pencil generated by M.T B M and A", "library": "Matplotlib plus NumPy", "artifact": "figures/cayley-klein-circle-pencil.png", "inspection_target": "finite distance circles are conics from one pencil; the absolute is the infinite-radius member", "validation": "each plotted circle is obtained from the circle-through-point matrix formula"},
    {"concept": "relation to the fundamental conic", "representation": "outside center, polar line, exceptional points, and tangent residuals", "library": "Matplotlib plus homogeneous matrix checks", "artifact": "figures/fundamental-conic-double-contact.png", "inspection_target": "all finite circles around the same center touch the absolute at the polar intersections", "validation": "circle and absolute tangent lines are parallel at both contact points"},
    {"concept": "centers at infinity", "representation": "Euclidean centers m=(a,b,c) with c tending to zero", "library": "Matplotlib plus matrix-limit residual", "artifact": "figures/center-at-infinity-limit.png", "inspection_target": "large Euclidean circles flatten to a finite line plus the line at infinity", "validation": "normalized conic matrix converges to the source-span limit matrix"},
    {"concept": "organizing principles for cycles", "representation": "classification panel and definition dependency graph", "library": "Matplotlib and NetworkX", "artifact": "figures/cycle-classification-panel.png; figures/cycle-organizing-principles.png", "inspection_target": "circle, horocycle, and hypercycle differ by center position and real contact pattern", "validation": "contact counts and graph nodes are recorded in visual checks"},
    {"concept": "cycles in Galilean geometry", "representation": "vertical-line distance circles, parabolic cycles, and theorem probes on y=x^2", "library": "Matplotlib, Pandas, SymPy", "artifact": "figures/galilean-circles-and-cycles.png; figures/galilean-cycle-theorem-lab.png; tables/galilean-cycle-measurements.csv", "inspection_target": "Galilean distance uses x-coordinate difference, while cycle theorems live on vertical parabolas", "validation": "slope-difference and secant-product identities are checked exactly and numerically"},
    {"concept": "Galilean cycle as a deformation limit", "representation": "Plotly slider through conics with ideal quadratic part x^2 + k y^2", "library": "Plotly HTML", "artifact": "html/galilean-deformation-lab.html", "inspection_target": "hyperbola-like, parabola, and ellipse-like cycles meet at k=0 as the ideal points coalesce", "validation": "every slider conic passes through the same three finite points"},
]

storyboard_path = save_json({
    "chapter": 23,
    "title": "Circles and Cycles",
    "source_span": {"printed_pages": "443-464", "pdf_pages": "465-486", "sections": "23.1-23.5"},
    "library_routing": {"matplotlib": "durable 2D conic diagrams and theorem probes", "plotly": "interactive one-parameter deformation of cycles", "sympy": "exact Galilean angle and secant-product identities", "networkx": "organizing-principles dependency graph", "course_utils": "artifact display and homogeneous-coordinate helpers"},
    "items": STORYBOARD,
}, ARTIFACT_ROOT, "checks", "storyboard.json")
storyboard_path.relative_to(BOOK_ROOT)

## 1. Distance Circles Are Conics in a Pencil

For a fixed center `m`, the expression `(m x p).T B (m x p)` is quadratic in the moving point `p`. Setting it proportional to `p.T A p` therefore gives a conic. The first figure draws several such conics for the same center in the hyperbolic unit disk model. The black circle is the absolute; it is the infinite-radius member of the pencil. The finite members are not guessed from Euclidean shape: each is computed from a point that it must pass through.

In [ ]:
m_inside = hpoint(0.18, -0.16)
probe_points = [hpoint(0.18 + d, -0.16 + 0.08 * i) for i, d in enumerate([0.18, 0.36, 0.55])]
family_matrices = [circle_through_point(A_HYP, B_HYP, m_inside, p) for p in probe_points]
M_inside = skew_cross_matrix(m_inside)
zero_radius_inside = M_inside.T @ B_HYP @ M_inside
psi_values = [float((p @ zero_radius_inside @ p) / ((m_inside @ A_HYP @ m_inside) * (p @ A_HYP @ p))) for p in probe_points]

fig, ax = plt.subplots(figsize=(6.5, 6.1))
draw_unit_absolute(ax)
colors = ["#2c7fb8", "#f03b20", "#31a354"]
proxies = [plt.Line2D([0], [0], color="#222222", lw=1.8, label="absolute / radius infinity")]
for idx, (C, p, color) in enumerate(zip(family_matrices, probe_points, colors), start=1):
    _, proxy = draw_conic(ax, C, (-1.1, 1.1), (-1.1, 1.1), color, label=f"distance member {idx}")
    proxies.append(proxy)
    ax.scatter([p[0]], [p[1]], color=color, s=24, zorder=5)
ax.scatter([m_inside[0]], [m_inside[1]], marker="*", color="#111111", s=80, zorder=6)
ax.text(m_inside[0] + 0.03, m_inside[1] - 0.05, "center m", color="#111111")
ax.set_title("Distance from one center produces a conic pencil")
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("affine x")
ax.set_ylabel("affine y")
ax.legend(handles=proxies, loc="upper right", frameon=False)
ax.grid(alpha=0.15)
pencil_path = save_current_figure(FIG_DIR / "cayley-klein-circle-pencil.png")
CHECKS["distance_pencil_psi_values"] = psi_values
display_artifact(pencil_path, width=720)

## 2. The Fundamental Conic Supplies the Contact Points

When the center is outside a nondegenerate real absolute, the exceptional points are real. They are the intersections of the polar of `m` with the absolute. The next figure uses an outside center so the contacts can be inspected directly. The finite circle, the zero-radius tangent pair, and the absolute all share the same two boundary points; the code checks this by comparing tangent lines at those points.

In [ ]:
m_outside = hpoint(1.46, 0.34)
p_for_hypercycle = hpoint(-0.22, 0.24)
hypercycle_C = circle_through_point(A_HYP, B_HYP, m_outside, p_for_hypercycle)
zero_radius_outside = circle_pencil_member(A_HYP, B_HYP, m_outside, lam=1, mu=0)
polar_line = A_HYP @ m_outside
contact_points = unit_circle_line_intersections(polar_line)
contact_residual = max(abs(float(e @ hypercycle_C @ e)) for e in contact_points)
absolute_residual = max(abs(float(e @ A_HYP @ e)) for e in contact_points)
tangent_residual = max(tangent_parallel_residual(hypercycle_C, A_HYP, e) for e in contact_points)

fig, ax = plt.subplots(figsize=(7.0, 5.8))
draw_unit_absolute(ax)
_, finite_proxy = draw_conic(ax, hypercycle_C, (-1.25, 1.75), (-1.25, 1.25), "#d95f02", label="finite circle / hypercycle", lw=2.2)
_, zero_proxy = draw_conic(ax, zero_radius_outside, (-1.25, 1.75), (-1.25, 1.25), "#1b9e77", label="zero-radius tangent pair", lw=1.4, alpha=0.75, linestyle="--")
draw_line(ax, polar_line, (-1.25, 1.75), color="#5e3c99", lw=1.4, linestyle=":", label="polar of m")
for e in contact_points:
    ax.scatter([e[0]], [e[1]], s=50, facecolor="white", edgecolor="#111111", zorder=6)
    ax.plot([m_outside[0], e[0]], [m_outside[1], e[1]], color="#1b9e77", lw=1.1, alpha=0.55)
ax.scatter([m_outside[0]], [m_outside[1]], marker="*", color="#111111", s=85, zorder=7)
ax.text(m_outside[0] + 0.03, m_outside[1] + 0.04, "center outside")
ax.set_xlim(-1.2, 1.68)
ax.set_ylim(-1.15, 1.15)
ax.set_aspect("equal", adjustable="box")
ax.set_title("All circles around an outside center touch the absolute twice")
ax.set_xlabel("affine x")
ax.set_ylabel("affine y")
ax.grid(alpha=0.15)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles + [finite_proxy, zero_proxy], loc="lower right", frameon=False)
contact_path = save_current_figure(FIG_DIR / "fundamental-conic-double-contact.png")
CHECKS["contact_value_residual"] = float(contact_residual)
CHECKS["contact_absolute_residual"] = float(absolute_residual)
CHECKS["contact_tangent_residual"] = float(tangent_residual)
display_artifact(contact_path, width=760)

## 3. Moving the Center to Infinity

In the standard Euclidean Cayley-Klein embedding, the primal absolute is the doubled line at infinity and the dual absolute stores the two circular points. A homogeneous center `m=(a,b,c)` moves to infinity when `c` tends to zero. Keeping one finite point on the circle fixed, the finite trace flattens to the line through that point with normal `(a,b)`. Projectively, the missing component is the line at infinity.

In [ ]:
p_fixed = hpoint(-0.35, 0.55)
direction = np.array([1.0, 0.55])
cs = [1.0, 0.48, 0.20, 0.08]
theta = np.linspace(0, 2 * np.pi, 1200)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
limit_xs = np.linspace(-3.2, 3.2, 300)
a, b = direction
limit_ys = p_fixed[1] - (a / b) * (limit_xs - p_fixed[0])
ax.plot(limit_xs, limit_ys, color="#111111", lw=2.0, label="finite limit line")
for c, color in zip(cs, ["#9ecae1", "#6baed6", "#3182bd", "#08519c"]):
    center = direction / c
    radius = np.linalg.norm(center - p_fixed[:2])
    xs = center[0] + radius * np.cos(theta)
    ys = center[1] + radius * np.sin(theta)
    ax.plot(xs, ys, color=color, lw=1.8, label=f"c={c:g}")
ax.scatter([p_fixed[0]], [p_fixed[1]], color="#d95f02", s=45, zorder=5, label="fixed point p")
ax.arrow(p_fixed[0], p_fixed[1], -b * 0.42, a * 0.42, width=0.01, color="#555555", length_includes_head=True)
ax.text(p_fixed[0] - 0.55, p_fixed[1] + 0.58, "limit direction")
ax.set_xlim(-3.0, 3.0)
ax.set_ylim(-2.6, 2.6)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Euclidean circles with center tending to infinity")
ax.set_xlabel("affine x")
ax.set_ylabel("affine y")
ax.grid(alpha=0.18)
ax.legend(loc="upper right", frameon=False)
center_limit_path = save_current_figure(FIG_DIR / "center-at-infinity-limit.png")
limit_C = np.array([[0.0, 0.0, -a], [0.0, 0.0, -b], [-a, -b, 2 * a * p_fixed[0] + 2 * b * p_fixed[1]]])
eps = 1e-5
C_eps = circle_through_point(A_EUC, B_EUC, np.array([a, b, eps]), p_fixed) / eps
CHECKS["center_infinity_matrix_residual"] = float(np.linalg.norm(normalize_matrix(C_eps) - normalize_matrix(limit_C)))
display_artifact(center_limit_path, width=760)

## 4. Organizing Principles: Circle, Horocycle, Hypercycle

The chapter broadens the word circle into the more flexible word cycle. The classification below uses the hyperbolic absolute as a readable test case. A finite-center circle has no real boundary contact. A horocycle has a boundary center and one real contact counted with higher multiplicity. A hypercycle has an outside center and two real contacts. The dependency graph next to it records why there is no single universal definition: distance, limiting behavior, dual angle, and curvature agree in some geometries and split in others.

In [ ]:
horocycle_C = np.array([[1.0, 0.0, -0.55], [0.0, 1.0, 0.0], [-0.55, 0.0, 0.10]])
classification = [("finite circle", family_matrices[1], [], "center inside"), ("horocycle", horocycle_C, [hpoint(1.0, 0.0)], "center on absolute"), ("hypercycle", hypercycle_C, contact_points, "center outside")]
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.8), sharex=True, sharey=True)
for ax, (title, C, contacts, subtitle) in zip(axes, classification):
    draw_unit_absolute(ax)
    draw_conic(ax, C, (-1.12, 1.12), (-1.12, 1.12), "#386cb0", lw=2.1)
    for e in contacts:
        ax.scatter([e[0]], [e[1]], s=42, facecolor="white", edgecolor="#bf5b17", zorder=6)
    ax.set_title(f"{title}\n{subtitle}")
    ax.set_xlim(-1.12, 1.12)
    ax.set_ylim(-1.12, 1.12)
    ax.grid(alpha=0.12)
axes[0].set_ylabel("affine y")
for ax in axes:
    ax.set_xlabel("affine x")
fig.suptitle("Cycle types organized by contact with the fundamental conic", y=1.02)
classification_path = save_current_figure(FIG_DIR / "cycle-classification-panel.png")
CHECKS["classification_real_contact_counts"] = {title: len(contacts) for title, _, contacts, _ in classification}

definition_graph = nx.DiGraph()
definition_graph.add_edges_from([("absolute conic", "distance formula"), ("distance formula", "centered circle"), ("centered circle", "conic pencil"), ("conic pencil", "exceptional contacts"), ("conic pencil", "limit cycles"), ("limit cycles", "horocycles"), ("limit cycles", "Galilean cycles"), ("dual angle", "dual cycles"), ("constant curvature", "cycle definition"), ("Galilean cycles", "parabolas"), ("Galilean distance", "vertical line pairs")])
pos = {"absolute conic": (0.0, 0.8), "distance formula": (1.2, 0.8), "centered circle": (2.4, 0.8), "conic pencil": (3.6, 0.8), "exceptional contacts": (4.8, 1.15), "limit cycles": (4.8, 0.55), "horocycles": (6.0, 0.9), "Galilean cycles": (6.0, 0.35), "parabolas": (7.1, 0.35), "dual angle": (2.2, 0.05), "dual cycles": (3.4, 0.05), "constant curvature": (4.7, -0.05), "cycle definition": (5.9, -0.05), "Galilean distance": (5.8, -0.55), "vertical line pairs": (7.1, -0.55)}
fig, ax = plt.subplots(figsize=(11.6, 4.2))
nx.draw_networkx_edges(definition_graph, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, width=1.2, edge_color="#777777")
node_colors = ["#f6e8c3" if "Galilean" in n or "vertical" in n or "parabolas" in n else "#c7eae5" for n in definition_graph.nodes]
nx.draw_networkx_nodes(definition_graph, pos, ax=ax, node_color=node_colors, node_size=1650, edgecolors="#333333", linewidths=0.8)
nx.draw_networkx_labels(definition_graph, pos, ax=ax, font_size=8)
ax.set_title("Organizing principles: which definition is doing the work?")
ax.axis("off")
organizing_path = save_current_figure(FIG_DIR / "cycle-organizing-principles.png")
CHECKS["organizing_graph_nodes"] = int(definition_graph.number_of_nodes())
CHECKS["organizing_graph_edges"] = int(definition_graph.number_of_edges())
display_artifact(classification_path, width=900)
display_artifact(organizing_path, width=900)

## 5. Galilean Geometry: Circles Split from Cycles

In the Galilean standard embedding used here, finite points have oriented distance `dist(p,q)=x_p-x_q`. Holding distance from a finite center fixed therefore gives two vertical lines. That is too small to serve as the full cycle theory. The limiting conic viewpoint instead produces vertical parabolas. The next figures separate these two facts and then test two Galilean analogues of familiar circle theorems on the standard cycle `y=x^2`.

In [ ]:
g_center = np.array([0.25, 0.45])
g_point = np.array([1.15, -0.25])
g_distance = g_point[0] - g_center[0]
xs = np.linspace(-2.1, 2.1, 500)
cycle_y = 0.62 * (xs + 0.42) ** 2 - 0.7
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.4))
ax = axes[0]
for xline, label in [(g_center[0] + abs(g_distance), "same x-distance"), (g_center[0] - abs(g_distance), "opposite side")]:
    ax.axvline(xline, color="#2b8cbe", lw=2.2, label=label)
ax.scatter([g_center[0]], [g_center[1]], marker="*", color="#111111", s=80, zorder=5, label="center")
ax.scatter([g_point[0]], [g_point[1]], color="#d95f02", s=45, zorder=5, label="probe point")
ax.set_title("Distance circle: two vertical lines")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-1.4, 2.5)
ax.grid(alpha=0.16)
ax.legend(frameon=False, loc="upper left")
ax = axes[1]
ax.plot(xs, cycle_y, color="#756bb1", lw=2.3, label="Galilean cycle")
ax.axvline(-0.42, color="#555555", lw=1.2, linestyle="--", label="vertical symmetry axis")
ax.set_title("Cycle: vertical parabola")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-1.4, 2.5)
ax.grid(alpha=0.16)
ax.legend(frameon=False, loc="upper left")
fig.suptitle("Galilean distance circles are a special degenerate part of the cycle family", y=1.03)
galilean_path = save_current_figure(FIG_DIR / "galilean-circles-and-cycles.png")

def parabola_point(t):
    return np.array([t, t * t], dtype=float)

a_param, b_param = -1.25, 1.05
A_pt, B_pt = parabola_point(a_param), parabola_point(b_param)
x_samples = [-0.65, 0.15, 1.65]
angle_rows = []
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6))
ax = axes[0]
plot_xs = np.linspace(-1.8, 1.9, 500)
ax.plot(plot_xs, plot_xs**2, color="#222222", lw=1.8, label="y=x^2")
ax.scatter([A_pt[0], B_pt[0]], [A_pt[1], B_pt[1]], color=["#1b9e77", "#d95f02"], s=48, zorder=5)
ax.text(A_pt[0] - 0.12, A_pt[1] + 0.18, "A")
ax.text(B_pt[0] + 0.05, B_pt[1] + 0.15, "B")
for t, color in zip(x_samples, ["#6baed6", "#3182bd", "#08519c"]):
    X_pt = parabola_point(t)
    slope_AX = (A_pt[1] - X_pt[1]) / (A_pt[0] - X_pt[0])
    slope_BX = (B_pt[1] - X_pt[1]) / (B_pt[0] - X_pt[0])
    angle_rows.append({"kind": "peripheral_angle", "parameter": t, "value": slope_AX - slope_BX})
    ax.plot([X_pt[0], A_pt[0]], [X_pt[1], A_pt[1]], color=color, lw=1.0, alpha=0.72)
    ax.plot([X_pt[0], B_pt[0]], [X_pt[1], B_pt[1]], color=color, lw=1.0, alpha=0.72)
    ax.scatter([X_pt[0]], [X_pt[1]], color=color, s=32, zorder=5)
ax.set_title("Peripheral angle becomes slope difference")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-1.8, 1.9)
ax.set_ylim(-0.4, 3.5)
ax.grid(alpha=0.16)
P = np.array([0.35, -0.45], dtype=float)
slopes = [-2.4, -1.4, 2.5, 3.2]
secant_rows = []
ax = axes[1]
ax.plot(plot_xs, plot_xs**2, color="#222222", lw=1.8, label="y=x^2")
ax.scatter([P[0]], [P[1]], marker="*", color="#111111", s=80, zorder=6)
ax.text(P[0] + 0.05, P[1] - 0.15, "P")
for slope, color in zip(slopes, ["#9ecae1", "#6baed6", "#3182bd", "#08519c"]):
    roots = np.sort(np.real_if_close(np.roots([1.0, -slope, slope * P[0] - P[1]])).astype(float))
    product = (roots[0] - P[0]) * (roots[1] - P[0])
    secant_rows.append({"kind": "secant_product", "parameter": slope, "value": product})
    ax.plot(plot_xs, P[1] + slope * (plot_xs - P[0]), color=color, lw=1.15, alpha=0.75)
    pts = np.array([parabola_point(r) for r in roots])
    ax.scatter(pts[:, 0], pts[:, 1], color=color, s=28, zorder=5)
ax.set_title("Secant product depends only on P")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-1.8, 1.9)
ax.set_ylim(-0.8, 3.8)
ax.grid(alpha=0.16)
fig.suptitle("Two Galilean cycle theorems as measurable checks", y=1.03)
theorem_path = save_current_figure(FIG_DIR / "galilean-cycle-theorem-lab.png")
measurement_rows = angle_rows + secant_rows
table_path = save_table(measurement_rows, ARTIFACT_ROOT, "tables", "galilean-cycle-measurements.csv")
ARTIFACTS.append(table_path)
CHECKS["galilean_distance_probe"] = float(g_distance)
CHECKS["galilean_peripheral_angle_spread"] = float(np.ptp([row["value"] for row in angle_rows]))
CHECKS["galilean_secant_product_spread"] = float(np.ptp([row["value"] for row in secant_rows]))
CHECKS["galilean_secant_expected"] = float(P[0] ** 2 - P[1])
display_artifact(galilean_path, width=850)
display_artifact(theorem_path, width=850)
display(pd.DataFrame(measurement_rows))

## 6. Applied Deformation Lab

## Applied Lab: Deforming Cycles Through Three Points

The interactive HTML lab below keeps three finite points fixed and changes the ideal quadratic part of the conic. The family

`x^2 + k y^2 + u x + v y + w = 0`

passes through the same three finite points for every value of `k`. For negative `k` the conic has two real ideal directions, for positive `k` those ideal directions are complex, and at `k=0` they coalesce into the Galilean point at infinity. In the finite chart this middle case is a vertical parabola. Move the slider in the HTML artifact and inspect the moment where the shape changes type.

In [ ]:
fixed_points = np.array([[-1.25, 1.10], [0.25, -0.15], [1.25, 1.55]], dtype=float)
lab_ks = [-1.0, -0.55, -0.18, 0.0, 0.18, 0.55, 1.0]
lab_xs = np.linspace(-2.2, 2.2, 150)
lab_ys = np.linspace(-2.0, 2.6, 150)
xx, yy = np.meshgrid(lab_xs, lab_ys)

def deformation_coefficients(k):
    matrix = np.column_stack([fixed_points[:, 0], fixed_points[:, 1], np.ones(len(fixed_points))])
    rhs = -(fixed_points[:, 0] ** 2 + k * fixed_points[:, 1] ** 2)
    return np.linalg.solve(matrix, rhs)

traces = []
for idx, k in enumerate(lab_ks):
    u, v, w = deformation_coefficients(k)
    zz = xx**2 + k * yy**2 + u * xx + v * yy + w
    traces.append(go.Contour(x=lab_xs, y=lab_ys, z=zz, contours={"start": 0, "end": 0, "size": 0.02, "coloring": "lines"}, line={"width": 3}, showscale=False, visible=(idx == 3), name=f"k={k:g}"))
traces.append(go.Scatter(x=fixed_points[:, 0], y=fixed_points[:, 1], mode="markers+text", text=["P", "Q", "R"], textposition="top center", marker={"size": 10, "color": "#d95f02"}, name="fixed finite points", visible=True))
steps = []
for idx, k in enumerate(lab_ks):
    visible = [False] * len(lab_ks) + [True]
    visible[idx] = True
    steps.append({"method": "update", "label": f"{k:g}", "args": [{"visible": visible}, {"title": f"Cycle deformation: k={k:g}"}]})
fig = go.Figure(data=traces)
fig.update_layout(title="Cycle deformation: k=0", width=760, height=560, xaxis={"scaleanchor": "y", "title": "x", "range": [-2.2, 2.2]}, yaxis={"title": "y", "range": [-2.0, 2.6]}, sliders=[{"active": 3, "currentvalue": {"prefix": "ideal parameter k = "}, "steps": steps}], margin={"l": 50, "r": 30, "t": 60, "b": 60})
html_path = HTML_DIR / "galilean-deformation-lab.html"
fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
ARTIFACTS.append(html_path)
max_fixed_residual = 0.0
for k in lab_ks:
    u, v, w = deformation_coefficients(k)
    vals = fixed_points[:, 0] ** 2 + k * fixed_points[:, 1] ** 2 + u * fixed_points[:, 0] + v * fixed_points[:, 1] + w
    max_fixed_residual = max(max_fixed_residual, float(np.max(np.abs(vals))))
CHECKS["deformation_fixed_point_residual"] = max_fixed_residual
display_artifact(html_path, width="100%", height=560)

## Direct Artifact Links

These links make the notebook readable before execution and give the audits stable artifact references.

![Cayley-Klein circle pencil](../../artifacts/chapter-23-circles-and-cycles/figures/cayley-klein-circle-pencil.png)

![Fundamental conic double contact](../../artifacts/chapter-23-circles-and-cycles/figures/fundamental-conic-double-contact.png)

![Center at infinity limit](../../artifacts/chapter-23-circles-and-cycles/figures/center-at-infinity-limit.png)

![Cycle classification panel](../../artifacts/chapter-23-circles-and-cycles/figures/cycle-classification-panel.png)

![Galilean theorem lab](../../artifacts/chapter-23-circles-and-cycles/figures/galilean-cycle-theorem-lab.png)

Open the [Galilean deformation lab](../../artifacts/chapter-23-circles-and-cycles/html/galilean-deformation-lab.html) and the [measurement table](../../artifacts/chapter-23-circles-and-cycles/tables/galilean-cycle-measurements.csv) for the parameter and numeric records.

## Proof and Invariant Scaffolds

The notebook uses three small proof scaffolds rather than a long formal proof.

First, the contact check is differential: at each exceptional point, the tangent line of the computed circle and the tangent line of the absolute are represented by proportional homogeneous line vectors. This is the matrix version of double contact.

Second, the center-at-infinity check is projective: the conic matrix for a Euclidean circle through a fixed point is divided by the vanishing homogeneous coordinate of the center. The normalized matrix converges to a degenerate conic whose finite component is the expected line.

Third, the Galilean theorem checks are symbolic. On `y=x^2`, the slope of the chord from parameter `a` to parameter `x` simplifies to `a+x`, so the difference of two such slopes is independent of `x`. For the secant theorem, the collinearity condition for a point `P=(x,y)` and two parabola parameters `a,b` is exactly the correction term that reduces `(a-x)(b-x)` to `x^2-y`.

In [ ]:
a_sym, b_sym, x_sym, y_sym = sp.symbols("a b x y")
peripheral_identity = sp.simplify((a_sym**2 - x_sym**2) / (a_sym - x_sym) - (b_sym**2 - x_sym**2) / (b_sym - x_sym) - (a_sym - b_sym))
secant_condition = a_sym * b_sym + y_sym - (a_sym + b_sym) * x_sym
secant_identity = sp.simplify((a_sym - x_sym) * (b_sym - x_sym) - (x_sym**2 - y_sym) - secant_condition)
assert peripheral_identity == 0
assert secant_identity == 0
beta = 0.37
m_dual = hpoint(0.32, -0.18)
m_star = A_HYP @ m_dual
X_primal = np.outer(m_star, m_star) + beta * A_HYP
beta_dual = -beta - float(m_dual @ A_HYP @ m_dual)
X_dual = np.outer(m_dual, m_dual) + beta_dual * np.linalg.inv(A_HYP)
dual_product = X_primal @ X_dual
dual_scalar = np.trace(dual_product) / 3.0
CHECKS["dual_circle_product_residual"] = float(np.linalg.norm(dual_product - dual_scalar * np.eye(3)))
sample = [-1.4, -0.2, 0.75, 1.6]
image = [(1.1 * t - 0.25) / (0.22 * t + 1.0) for t in sample]
CHECKS["cross_ratio_error"] = float(abs(cross_ratio(*sample) - cross_ratio(*image)))
{"peripheral_identity": str(peripheral_identity), "secant_identity": str(secant_identity), "dual_circle_product_residual": CHECKS["dual_circle_product_residual"], "cross_ratio_error": CHECKS["cross_ratio_error"]}

## Final Sanity Checks

The final cell records the artifact inventory, nonblank image statistics, symbolic identities, contact residuals, limit residuals, and Galilean numeric spreads. Future edits should fail here if a diagram is stale, if an artifact path moves, or if one of the chapter-specific invariants no longer matches the rendered construction.

In [ ]:
ARTIFACTS = sorted(set(Path(path) for path in ARTIFACTS), key=lambda p: p.as_posix())
assert_artifacts(ARTIFACTS, min_size=256)
image_summaries = []
for path in ARTIFACTS:
    if path.suffix.lower() == ".png":
        stats = image_stats(path)
        assert stats["width"] >= 200 and stats["height"] >= 150
        assert stats["pixel_std"] > 1.0
        image_summaries.append({"path": book_relative(path), "width": stats["width"], "height": stats["height"], "pixel_std": stats["pixel_std"], "file_size": stats["file_size"]})
assert CHECKS["contact_value_residual"] < 1e-8
assert CHECKS["contact_absolute_residual"] < 1e-10
assert CHECKS["contact_tangent_residual"] < 1e-8
assert CHECKS["center_infinity_matrix_residual"] < 1e-4
assert CHECKS["galilean_peripheral_angle_spread"] < 1e-12
assert CHECKS["galilean_secant_product_spread"] < 1e-12
assert CHECKS["deformation_fixed_point_residual"] < 1e-10
assert CHECKS["dual_circle_product_residual"] < 1e-10
assert CHECKS["cross_ratio_error"] < 1e-12
visual_checks = {"chapter": 23, "all_files_exist": all(path.exists() for path in ARTIFACTS), "artifacts": [book_relative(path) for path in ARTIFACTS], "cross_ratio_error": CHECKS["cross_ratio_error"], "numeric_checks": CHECKS, "image_summaries": image_summaries}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
final = {"chapter": 23, "source_span": {"printed_pages": "443-464", "pdf_pages": "465-486", "sections": "23.1-23.5"}, "notebook_executed": True, "artifacts": [book_relative(path) for path in ARTIFACTS], "symbolic_checks": {"galilean_peripheral_identity": str(peripheral_identity), "galilean_secant_identity": str(secant_identity)}, "numeric_checks": CHECKS}
save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final

## Takeaways

- A Cayley-Klein circle is naturally a conic because the constant-distance condition is quadratic in the moving point.
- The fundamental conic is not just context: finite circles around a center share exceptional contact points determined by the center's polar.
- A center at infinity is best handled projectively. In the Euclidean degeneration, the finite part of the limiting circle is a line and the other component is the line at infinity.
- The word cycle collects several compatible ideas: distance circles, limiting conics, dual angle definitions, and constant-curvature curves. They agree in some geometries and separate in degenerate ones.
- Galilean geometry is the main warning case. Finite-distance circles are vertical-line pairs, but the robust cycle objects are vertical parabolas, and familiar circle theorems survive as slope-difference and secant-product identities.